# SLM Harness: Real Invoice Dataset EDA & DuckDB Warehouse

**Objective**: Perform Exploratory Data Analysis (EDA) on the **50 Real Invoice Corpus** using **DuckDB**.

**Dataset Source**: [Katana ML Invoices Dataset on Hugging Face (`katanaml-org/invoices-donut-data-v1`)](https://huggingface.co/datasets/katanaml-org/invoices-donut-data-v1)
**Real Images**: Stored locally in `data/real_invoices/images/`

This notebook evaluates:
1. **Invoice text distribution**: Verifying OCR Markdown length fits within GLiNER's token span context.
2. **Tabular structure**: Inspecting HTML tables and line items for TabPFN evaluation.
3. **Experiment tracking**: Querying runs, hypotheses, and extraction metrics directly in SQL.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

db_path = Path("../data/slm_warehouse.duckdb")
conn = duckdb.connect(str(db_path))
print(f"Connected to DuckDB: {db_path.resolve()}")

## 1. High-Level Real Corpus Overview
Examine total volume, real dollar ranges, and date distributions across the 50 real invoices.

In [ ]:
conn.execute("""
SELECT 
    COUNT(*) AS total_invoices,
    MIN(invoice_date) AS earliest_date,
    MAX(invoice_date) AS latest_date,
    ROUND(SUM(total_amount), 2) AS gross_total_amount,
    ROUND(AVG(total_amount), 2) AS avg_invoice_amount,
    ROUND(MIN(total_amount), 2) AS min_amount,
    ROUND(MAX(total_amount), 2) AS max_amount
FROM raw_invoices;
""").df()

## 2. Document Length & Token Feasibility for GLiNER
Transformer backbones (e.g. DeBERTa in GLiNER) have a maximum context window of 512 tokens.
Let's inspect estimated tokens per invoice based on character length (`len / 4`).

In [ ]:
conn.execute("""
SELECT 
    COUNT(*) AS invoice_count,
    ROUND(AVG(LENGTH(ocr_markdown)), 0) AS avg_md_chars,
    ROUND(AVG(LENGTH(ocr_markdown) / 4.0), 0) AS est_avg_tokens,
    ROUND(MIN(LENGTH(ocr_markdown) / 4.0), 0) AS min_tokens_seen,
    ROUND(MAX(LENGTH(ocr_markdown) / 4.0), 0) AS max_tokens_seen,
    ROUND(AVG(LENGTH(ocr_html_tables)), 0) AS avg_html_chars
FROM raw_invoices;
""").df()

## 3. Real Vendors & Billing Breakdown

In [ ]:
conn.execute("""
SELECT 
    vendor_name,
    COUNT(*) AS invoice_count,
    ROUND(SUM(total_amount), 2) AS total_billed,
    ROUND(AVG(total_amount), 2) AS avg_invoice_value
FROM raw_invoices
GROUP BY vendor_name
ORDER BY total_billed DESC
LIMIT 10;
""").df()

## 4. Real Line Items Granularity (For TabPFN Tabular Testing)
Inspect top line items and pricing patterns extracted from the real documents.

In [ ]:
conn.execute("""
SELECT 
    item_description,
    COUNT(*) AS frequency,
    ROUND(AVG(quantity), 1) AS avg_qty,
    ROUND(AVG(unit_price), 2) AS avg_unit_price,
    ROUND(SUM(line_total), 2) AS cumulative_spend
FROM invoice_line_items
GROUP BY item_description
ORDER BY cumulative_spend DESC
LIMIT 10;
""").df()

## 5. Experiment Tracking Verification (In DuckDB)
Query the harness experiment table.

In [ ]:
conn.execute("""
SELECT 
    run_id,
    pipeline_stage,
    model_name,
    hypothesis,
    parameters_json,
    metrics_json,
    completed_at
FROM experiment_runs;
""").df()